# Notebook 5 — Matplotlib: Visualising Data

Welcome. By the end of this notebook you will be able to create, customise, and
combine publication-ready plots in Python using Matplotlib, the standard scientific
plotting library. You will draw line plots, scatter plots, histograms, and
multi-panel figures, and every example will be flavoured with neuroscience data.

*Neuroscience in plain English:* Throughout this notebook you will work with things
like membrane voltage (how electrically charged a neuron is), firing rates (how
many times per second a neuron sends a signal), and inter-spike intervals (the
gaps between those signals). All of these are just numbers, and Matplotlib is how
scientists turn those numbers into pictures.

Work through each item in order. Predict what each plot will look like **before**
you run the cell, then run and compare.


---
## Item 1 — Your First Plot (Line Plot)

> **By the end of this section you will be able to:** import Matplotlib, create a
> figure, draw a line through a set of points, and display the result. You will also
> understand what a figure and an axes object are and why the distinction matters.
> Everything below the core sections is optional on a first pass.

---

### What Matplotlib actually does

Matplotlib is a library that takes lists of numbers and turns them into pictures.
That is its entire job.

**Sketching on paper.** When you draw a graph by hand, you first pick up a blank
sheet (the canvas), draw a box on it (the plotting area), label the edges, and then
draw your curve inside the box. Matplotlib does exactly these steps, just in code.

**A blank whiteboard in a lecture hall.** Before the lecturer draws anything, the
board is just potential. The board is the *figure*. The marked-out rectangle they
draw their axes on is the *axes*. Everything goes inside those axes.

In Matplotlib:
- A **figure** is the whole canvas, the outermost window or image file.
- An **axes** (plural of the same word in Matplotlib's usage) is the rectangular
  plotting area inside the figure, complete with its own x-axis, y-axis, tick marks,
  and labels.

One figure can hold many axes (you will see this in Item 5). For now, the simplest
call `plt.plot(x, y)` creates both at once behind the scenes.

### The import convention

Every Matplotlib tutorial and paper in the world uses the same abbreviation:

```python
import matplotlib.pyplot as plt
```

`pyplot` is the submodule that provides the MATLAB-style shorthand functions
(`plt.plot`, `plt.show`, `plt.figure`). The alias `plt` is universal. Using anything
else will confuse every colleague who reads your code.

### The three-step pattern

Almost every simple plot follows this pattern:

1. **Set up the canvas.** `plt.figure()` creates a new figure. You can pass
   `figsize=(width_in_inches, height_in_inches)` to control size.
2. **Draw something.** `plt.plot(x, y)` draws a line connecting the (x, y) points
   you supply.
3. **Display it.** `plt.show()` sends the finished figure to the screen (or, in
   Colab, renders it inline).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Worked Example 1a: the simplest possible line plot ---

# A plain list of x values and y values (same length required)
x = [0, 1, 2, 3, 4]        # five time points (imagine milliseconds)
y = [0, 1, 4, 9, 16]       # y = x^2, just to have a curve

plt.figure()                 # create a blank canvas
plt.plot(x, y)               # draw a line through the (x, y) points
plt.show()                   # display the result


Now let us do something real: plot a simulated membrane voltage trace.

*Membrane voltage (also called membrane potential):* A neuron is like a tiny
battery. The inside is about 70 millivolts (mV) more negative than the outside
when the cell is resting (-70 mV). When it fires a signal (an "action potential"
or "spike"), the voltage briefly shoots up to about +40 mV and then snaps back.
We will simulate this.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Worked Example 1b: simulated membrane voltage trace ---

# Create a time axis: 0 to 100 ms in steps of 0.1 ms
t = np.linspace(0, 100, 1000)   # 1000 evenly spaced points from 0 to 100

# Simulate resting voltage with a small amount of random noise
# A real neuron sits at around -70 mV; noise represents natural fluctuation
rng = np.random.default_rng(seed=42)   # seed fixes the randomness so results reproduce
voltage = -70 + rng.normal(0, 1.5, size=len(t))  # -70 mV plus small Gaussian noise

# Simulate three action potentials (spikes) at t = 20, 50, 80 ms
# A spike is a rapid voltage excursion: rises fast, falls fast
for spike_time in [20, 50, 80]:
    mask = (t >= spike_time) & (t <= spike_time + 2)  # 2 ms window for the spike
    voltage[mask] = 40 * np.exp(-((t[mask] - spike_time - 0.5) ** 2) / 0.2)

plt.figure(figsize=(10, 4))     # wider than tall suits a time series
plt.plot(t, voltage)             # x = time in ms, y = voltage in mV
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Worked Example 1c: spike count vs trial number ---
# "Trial" = one attempt in an experiment (e.g. one flash of a stimulus)
# "Spike count" = how many times a neuron fired during that trial

rng = np.random.default_rng(seed=7)
num_trials = 20
trial_numbers = np.arange(1, num_trials + 1)  # 1, 2, 3, ..., 20

# Simulate: the neuron fires more as trials progress (learning / adaptation)
spike_counts = rng.poisson(lam=trial_numbers / 4)  # Poisson: the natural model for counts

plt.figure(figsize=(8, 4))
plt.plot(trial_numbers, spike_counts)   # line connecting trial-by-trial counts
plt.show()


---

> ## Going deeper (optional on a first pass)
>
> You can skip this block the first time through and lose nothing essential.
>
> **The object-oriented interface.** The `plt.plot()` style is called the
> "state-machine" or "pyplot" interface: Matplotlib keeps track of a hidden
> "current figure" and "current axes" and applies each call to them. It is
> convenient for quick plots.
>
> The more powerful style is the object-oriented (OO) interface, where you hold
> explicit references to the figure and axes:
>
> ```python
> fig, ax = plt.subplots()   # returns the figure object AND the axes object
> ax.plot(x, y)              # call methods on the axes directly
> fig.savefig("output.png")  # save via the figure object
> ```
>
> You will use this style in Item 5. It becomes essential when you have multiple
> axes, because you can target each one by name rather than relying on "current."
>
> **`plt.savefig("name.png")`** saves the current figure to a file instead of (or
> in addition to) showing it. Call it before `plt.show()`. Supported formats include
> PNG, PDF, SVG, and JPEG.
>
> **DPI.** `plt.figure(dpi=150)` increases the pixel density. 150 or 300 dpi is
> common for publication figures.
>
> **Closing figures.** `plt.close()` clears the hidden current-figure state. In
> loops that create many figures, always close each one or memory fills up.
> `plt.close('all')` closes everything.
>
> **`np.linspace(start, stop, N)` vs `np.arange(start, stop, step)`.**
> `linspace` gives you exactly N evenly spaced points including both endpoints.
> `arange` gives points at a fixed step and may or may not include the stop. For
> time axes with a known number of samples, `linspace` is safer.

---

### Common questions and confusions

**"Do I need `plt.figure()` before every plot?"** Not strictly. `plt.plot()` will
create one automatically if none exists. But calling `plt.figure()` explicitly makes
your intent clear and lets you set `figsize` and `dpi`. Get into the habit.

**"What is the difference between `plt.plot(y)` and `plt.plot(x, y)`?"** When you
give only one argument, Matplotlib uses the index (0, 1, 2, ...) as the x-axis.
This works but hides what x actually represents. Always supply both `x` and `y`
when x has real meaning (time, trial number, etc.).

**"Why does `plt.show()` sometimes produce a blank window?"** In a plain Python
script outside Jupyter, the window appears and immediately closes when the script
ends. Put `plt.show()` at the very end, or add `input("Press Enter...")` after it.
In Jupyter / Colab, `plt.show()` is not strictly needed but is good practice.

**"I called `plt.plot()` twice and both lines appeared on the same axes. Is that a
bug?"** No, it is the intended behaviour. Repeated `plt.plot()` calls add to the
current axes. Use `plt.figure()` first if you want a fresh canvas.

---

### Your exercises

Predict what each plot will look like **before** you run the cell.

1. Create `x = [1, 2, 3, 4, 5]` and `y = [2, 4, 6, 8, 10]`. Plot the line. What
   shape do you expect?
2. Use `np.linspace(0, 2 * np.pi, 200)` as x and `np.sin(x)` as y. Plot the result.
   What wave should you see?
3. Call `plt.plot(x, y)` twice in the same cell (two different `y` arrays) without
   calling `plt.figure()` between them. What happens?
4. Change `figsize` to `(3, 6)` (portrait) and re-run Example 1b. How does the trace
   look different?
5. Add a third spike to the voltage simulation at `t = 65 ms` and re-plot.
6. *(Stretch.)* Plot two sine waves in the same figure: `np.sin(x)` and
   `np.sin(2 * x)`. Do they appear on the same axes? Now add a `plt.figure()` call
   between them. What changes?


In [ ]:
# Exercise 1
# Create x = [1, 2, 3, 4, 5] and y = [2, 4, 6, 8, 10], then plot

# your code here


In [ ]:
# Exercise 2
# Plot sin(x) for x from 0 to 2*pi

# your code here


In [ ]:
# Exercise 3
# Call plt.plot twice on the same axes without plt.figure() between them

# your code here


In [ ]:
# Exercise 4
# Change figsize to (3, 6) and re-run the voltage trace

# your code here


In [ ]:
# Exercise 5
# Add a third spike at t = 65 ms to the voltage trace

# your code here


In [ ]:
# Exercise 6 (Stretch)
# Plot sin(x) and sin(2x): first on the same axes, then on separate figures

# your code here


### The irreducible core

1. `import matplotlib.pyplot as plt` is the universal convention. Always use it.
2. A **figure** is the whole canvas; an **axes** is the drawing area inside it.
3. The basic pattern: `plt.figure()` -> `plt.plot(x, y)` -> `plt.show()`.
4. Multiple `plt.plot()` calls add lines to the **same** axes unless you open a new
   figure first.
5. `plt.figure(figsize=(w, h))` controls the canvas size in inches.

**You have got it when:** you can write a three-line script from memory that produces
a named line plot of any two same-length arrays, with the canvas sized to your
liking.


---
## Item 2 — Customising Plots

> **By the end of this section you will be able to:** add axis labels, a title, and
> a legend to any plot; choose colours, linestyles, linewidths, and transparency;
> set axis limits and add a grid; and annotate a neuroscience figure clearly enough
> that a reader can interpret it without asking you questions.

---

### Why customisation is not cosmetic

**Labelling a graph in a textbook.** A graph without axis labels is like a map
without a compass rose or scale bar: you can see the shape but you cannot read the
meaning. A reader who does not know that the y-axis is in millivolts cannot use your
figure.

**Dressing a display for clarity.** A museum exhibit designer does not leave objects
unlabelled and expect visitors to guess. Every label, colour choice, and layout
decision guides the viewer's eye to the important information. The same applies to
a scientific plot.

Labels and titles are not decoration. They are the difference between a figure that
communicates and one that does not.

### Axis labels and title

```python
plt.xlabel("Time (ms)")      # label for the x-axis, with units in parentheses
plt.ylabel("Voltage (mV)")   # label for the y-axis
plt.title("Membrane Voltage Trace")  # title at the top of the plot
```

Always include units in axis labels. "Time" without "(ms)" is ambiguous. "Voltage"
without "(mV)" is unpublishable.

### Legends

When you have more than one line, a **legend** identifies each. You give each line
a `label=` and then call `plt.legend()` to render the key.

```python
plt.plot(t, voltage_neuron_A, label="Neuron A")
plt.plot(t, voltage_neuron_B, label="Neuron B")
plt.legend()   # draws the key box with colour-coded labels
```

### Colours, linestyles, linewidth, alpha

These are keyword arguments to `plt.plot()`:

- **`color=`** accepts named colours (`"red"`, `"steelblue"`, `"orange"`) or hex
  codes (`"#2196F3"`). Named colours are fine for everyday use; hex gives exact
  control for publications.
- **`linestyle=`** (or `ls=`): `"-"` solid (default), `"--"` dashed, `":"` dotted,
  `"-."` dash-dot.
- **`linewidth=`** (or `lw=`): thickness in points. Default is 1.5. Use 2 or 2.5
  for presentation figures.
- **`alpha=`**: transparency from 0 (invisible) to 1 (opaque). Use 0.6-0.8 to let
  overlapping lines show through each other.

### Axis limits and grid

```python
plt.xlim(0, 100)   # set x-axis range
plt.ylim(-80, 50)  # set y-axis range; choose to include some headroom
plt.grid(True)     # add a faint grid to make values easier to read
plt.grid(True, linestyle="--", alpha=0.4)  # softer grid: dashed and semi-transparent
```

### Horizontal and vertical reference lines

`plt.axhline(y=value)` draws a horizontal line across the whole axes at a fixed y
value. This is perfect for threshold lines in neuroscience.

```python
plt.axhline(y=-55, color="red", linestyle="--", linewidth=1.5, label="Threshold")
```

*Action potential threshold:* Around -55 mV, the neuron's voltage reaches a point
of no return and the spike fires automatically. Marking this on the plot is standard
in neuroscience figures.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Worked Example 2a: fully labelled voltage trace with two neurons ---

rng = np.random.default_rng(seed=0)
t = np.linspace(0, 100, 1000)   # time axis: 0 to 100 ms

# Simulate two different neurons with different noise levels
voltage_A = -70 + rng.normal(0, 1.5, size=len(t))
voltage_B = -70 + rng.normal(0, 3.0, size=len(t))  # noisier neuron

# Add a spike to neuron A at t = 40 ms
for spike_time in [40, 75]:
    mask = (t >= spike_time) & (t <= spike_time + 2)
    voltage_A[mask] = 40 * np.exp(-((t[mask] - spike_time - 0.5) ** 2) / 0.2)

# Add a spike to neuron B at t = 60 ms only
mask_B = (t >= 60) & (t <= 62)
voltage_B[mask_B] = 35 * np.exp(-((t[mask_B] - 60.5) ** 2) / 0.2)

plt.figure(figsize=(11, 4))

# Plot each neuron with distinct colour, linestyle, and label
plt.plot(t, voltage_A,
         color="steelblue",    # a calm, professional blue
         linewidth=1.8,
         linestyle="-",        # solid line for Neuron A
         label="Neuron A")

plt.plot(t, voltage_B,
         color="coral",        # warm orange-red for contrast
         linewidth=1.8,
         linestyle="--",       # dashed for Neuron B, visually distinct
         alpha=0.85,           # slight transparency so lines don't fight
         label="Neuron B")

# Threshold line: -55 mV is where a spike becomes inevitable
plt.axhline(y=-55,
            color="crimson",
            linestyle=":",
            linewidth=1.5,
            label="Spike threshold (-55 mV)")

# Labels and legend
plt.xlabel("Time (ms)")          # always include units
plt.ylabel("Membrane voltage (mV)")
plt.title("Two Neurons: Membrane Voltage Over Time")
plt.legend(loc="upper right")    # loc places the legend; "best" also works
plt.xlim(0, 100)
plt.ylim(-85, 55)
plt.grid(True, linestyle="--", alpha=0.3)   # soft grid in the background
plt.tight_layout()    # adjusts spacing so labels don't get clipped
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Worked Example 2b: firing rate across conditions with clear labels ---
# "Firing rate" = average number of spikes per second (Hz = Hertz = per second)
# "Condition" = experimental setting (e.g. low vs high stimulus intensity)

rng = np.random.default_rng(seed=3)
time_points = np.linspace(0, 500, 50)   # 0 to 500 ms, 50 sample points

# Simulate firing rate time courses under two stimulus conditions
rate_low  = 5  + 10 * np.exp(-time_points / 150) + rng.normal(0, 0.8, 50)
rate_high = 5  + 30 * np.exp(-time_points / 150) + rng.normal(0, 1.5, 50)

plt.figure(figsize=(9, 4))

plt.plot(time_points, rate_low,
         color="#4C72B0",       # hex colour: a muted blue
         linewidth=2,
         label="Low stimulus (10% contrast)")

plt.plot(time_points, rate_high,
         color="#DD8452",       # hex colour: warm amber
         linewidth=2,
         label="High stimulus (80% contrast)")

plt.axhline(y=5, color="grey", linestyle=":", linewidth=1.2,
            label="Baseline rate (5 Hz)")

plt.xlabel("Time after stimulus onset (ms)")
plt.ylabel("Firing rate (Hz)")
plt.title("Population Firing Rate: Low vs High Stimulus Contrast")
plt.legend()
plt.xlim(0, 500)
plt.ylim(0, 45)
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()


---

> ## Going deeper (optional on a first pass)
>
> **Colour specification options.** Beyond named colours and hex, Matplotlib accepts:
> RGB tuples `(0.2, 0.5, 0.8)` where each channel is 0-1; RGBA tuples that also
> set transparency `(0.2, 0.5, 0.8, 0.6)`; and single-letter shortcuts
> (`"b"` = blue, `"r"` = red, `"g"` = green, `"k"` = black, `"w"` = white).
>
> **Colour palettes.** For more than two or three lines, use a palette designed for
> distinguishability. `matplotlib.cm.tab10` and `seaborn`'s colour palettes are
> common choices. Avoid using red and green together (common form of colour
> blindness).
>
> **`plt.axvline(x=value)`** draws a vertical line at a fixed x, useful for marking
> event onset times (e.g. when a stimulus appeared).
>
> **`plt.text(x, y, "string")`** places a text annotation at a data-coordinate
> position. `plt.annotate()` does the same with an optional arrow.
>
> **`plt.tick_params()`** controls tick mark size, direction, and which sides they
> appear on. Useful for publication figures.
>
> **`plt.rcParams`** is a global dictionary of default style settings. Changing
> `plt.rcParams["font.size"] = 14` before any plotting command raises the default
> font size for all subsequent figures.
>
> **Style sheets.** `plt.style.use("seaborn-v0_8-whitegrid")` applies a complete
> visual theme. Other popular choices: `"ggplot"`, `"bmh"`, `"dark_background"`.
> List all available styles with `print(plt.style.available)`.
>
> **Spine removal.** Many clean publication figures remove the top and right "box"
> lines: `ax.spines["top"].set_visible(False)` and
> `ax.spines["right"].set_visible(False)`. This requires the OO interface (see Item
> 1 Going Deeper).

---

### Common questions and confusions

**"My legend shows the line but not the label."** You forgot `label="..."` in the
`plt.plot()` call. The legend can only show what you named.

**"The axis labels are cut off in the saved figure."** Call `plt.tight_layout()`
before `plt.savefig()`. It recalculates spacing to fit everything inside the canvas
boundary.

**"What is the difference between `alpha` in `plt.plot()` and in `plt.grid()`?"**
Same parameter, same meaning: 0 is invisible, 1 is opaque. Used in `plt.grid()` to
make the grid lines recede so they do not compete with your data.

**"How do I know which colour names are valid?"** Any CSS named colour works (there
are 148 of them). In a notebook, try `"steelblue"`, `"tomato"`, `"mediumseagreen"`,
`"goldenrod"`. When in doubt, use a hex code; it is unambiguous.

**"`plt.xlim()` seems to do nothing."** Make sure you call it after `plt.plot()`,
not before. The plot call can reset the limits if called after.

---

### Your exercises

1. Take Example 2a and change Neuron A's colour to `"#E63946"` and Neuron B's
   linestyle to `"-."`. Predict the result, then run.
2. Add a `plt.axvline(x=50, color="green", linestyle="--")` to mark the midpoint
   in time. What does it look like?
3. Set `plt.ylim(-100, 60)` on Example 2a. What changes about the figure?
4. Change the grid to `alpha=0.8`. How does it affect readability of the data?
5. In Example 2b, add a label to the horizontal baseline line and move the legend
   to `loc="upper right"`.
6. *(Stretch.)* Using one `plt.figure()`, plot three sine waves with frequencies
   1, 2, and 3 Hz (x from 0 to 1 second), each a different colour and linestyle.
   Add proper axis labels, a title, a legend for each frequency, and a grid. This
   is the format expected in any neuroscience paper.

---

### The irreducible core

1. Always label axes with **units**: `plt.xlabel("Time (ms)")`,
   `plt.ylabel("Voltage (mV)")`.
2. Use `label=` in each `plt.plot()` and call `plt.legend()` to generate the key.
3. Control appearance with `color=`, `linestyle=`, `linewidth=`, and `alpha=`.
4. `plt.axhline(y=value)` draws a horizontal reference line (threshold, baseline).
5. `plt.tight_layout()` before saving or showing prevents label clipping.

**You have got it when:** you can produce a two-line neuroscience figure with a
labelled threshold, correct axis labels with units, a legend, and a grid, and it
is clean enough to drop into a presentation.


In [ ]:
# Exercise 1
# Change Neuron A's colour to '#E63946' and Neuron B's linestyle to '-.'

# your code here


In [ ]:
# Exercise 2
# Add a vertical line at x = 50 ms to mark the midpoint

# your code here


In [ ]:
# Exercise 3
# Set ylim(-100, 60) on Example 2a

# your code here


In [ ]:
# Exercise 4
# Change grid alpha to 0.8 -- does the data become harder to see?

# your code here


In [ ]:
# Exercise 5
# Add label to the baseline axhline, move legend to upper right

# your code here


In [ ]:
# Exercise 6 (Stretch)
# Three sine waves: frequencies 1, 2, 3 Hz; different colour + linestyle each
# Full labels, legend, grid

# your code here


---
## Item 3 — Scatter Plots

> **By the end of this section you will be able to:** create a scatter plot with
> `plt.scatter()`, control point colour, size, and transparency, add a colour bar,
> and choose correctly between scatter and line plots for a given dataset.

---

### When to scatter, when to line

**Mapping patient outcomes vs dosage.** Imagine you have 200 patients and for each
you recorded their drug dose and their symptom score. You do not connect these points
with a line because the patients have no natural sequence: Patient 47 is not "after"
Patient 46 in any meaningful sense. You want to see the *cloud* of points and whether
there is a trend. That is a scatter plot.

**Charting exam scores vs study hours.** Each student is a point. "Do students who
study more score higher?" is a scatter plot question. The relationship between two
variables measured on independent subjects is exactly what scatter is designed to
show.

**The rule of thumb:**
- Use a **line plot** when x represents ordered, continuous time or sequence, and the
  order matters (voltage over time, firing rate over milliseconds).
- Use a **scatter plot** when each point is an independent observation and the order
  is arbitrary (one dot per neuron, one dot per trial, one dot per patient).

### `plt.scatter()` syntax

```python
plt.scatter(x, y,
            c=colours,   # colour: a single colour string or an array of values
            s=sizes,     # marker size in points^2 (default ~36)
            alpha=0.7)   # transparency
```

When `c=` is an **array of numbers** rather than a single colour string, Matplotlib
maps those numbers to a colour scale. Call `plt.colorbar()` immediately after to
show the scale bar.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Worked Example 3a: firing rate vs stimulus intensity ---
# "Stimulus intensity" = how strong the input signal is (e.g. brightness, loudness)
# Each dot is one neuron recorded in one experiment

rng = np.random.default_rng(seed=11)
n_neurons = 80

stimulus_intensity = rng.uniform(0, 100, n_neurons)   # % of maximum, 0-100
# Neurons tend to fire more when stimulated harder, with some scatter
firing_rate = 2 + 0.45 * stimulus_intensity + rng.normal(0, 8, n_neurons)
firing_rate = np.clip(firing_rate, 0, None)  # firing rate can't be negative

plt.figure(figsize=(7, 5))
plt.scatter(stimulus_intensity, firing_rate,
            color="steelblue",
            s=50,          # marker size
            alpha=0.7,     # semi-transparent so overlapping points are visible
            edgecolors="white",  # thin white border makes points crisper
            linewidths=0.5)

plt.xlabel("Stimulus intensity (%)")
plt.ylabel("Firing rate (Hz)")
plt.title("Firing Rate vs Stimulus Intensity (n=80 neurons)")
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Worked Example 3b: neuron positions coloured by firing rate ---
# Imagine recording from a 2D patch of cortex.
# Each neuron has an (x, y) position and a firing rate.
# We colour each dot by its firing rate to see spatial patterns.

rng = np.random.default_rng(seed=22)
n_neurons = 120

# Random positions on a 1 mm x 1 mm cortical patch
x_pos = rng.uniform(0, 1, n_neurons)   # x position in mm
y_pos = rng.uniform(0, 1, n_neurons)   # y position in mm

# Firing rate varies with position (a gradient, like a "hot spot")
firing_rate = 20 * np.exp(-((x_pos - 0.5)**2 + (y_pos - 0.5)**2) / 0.08)
firing_rate += rng.normal(0, 1, n_neurons)
firing_rate = np.clip(firing_rate, 0, None)

plt.figure(figsize=(7, 6))

scatter = plt.scatter(x_pos, y_pos,
                      c=firing_rate,       # c= an array -> maps to colour scale
                      s=70,
                      alpha=0.85,
                      cmap="plasma",       # colour map: low=dark purple, high=yellow
                      edgecolors="grey",
                      linewidths=0.3)

cbar = plt.colorbar(scatter)              # draws the colour scale bar
cbar.set_label("Firing rate (Hz)")        # label the colour bar

plt.xlabel("Cortical position x (mm)")
plt.ylabel("Cortical position y (mm)")
plt.title("Neuron Firing Rates Across a Cortical Patch")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Worked Example 3c: tuning curve ---
# A "tuning curve" shows how a neuron's firing rate depends on a stimulus property.
# Classic example: a neuron in visual cortex fires most to bars tilted at 45 degrees.
# "Orientation tuning" = the preferred angle of lines/edges for a visual neuron.

orientations = np.arange(0, 181, 15)      # 0, 15, 30, ... 180 degrees
preferred_orientation = 75                 # this neuron likes 75-degree tilts

# Firing rate peaks at the preferred orientation (Gaussian tuning)
tuning = 40 * np.exp(-((orientations - preferred_orientation)**2) / (2 * 30**2))
tuning += 5   # baseline firing rate (fires a little even to non-preferred stimuli)

rng = np.random.default_rng(seed=5)
tuning_noisy = tuning + rng.normal(0, 2, len(orientations))  # add measurement noise

plt.figure(figsize=(8, 5))

# Plot the underlying smooth tuning curve as a line
plt.plot(orientations, tuning, color="steelblue", linewidth=2,
         linestyle="--", label="True tuning curve", alpha=0.6)

# Plot the noisy measured data as scatter (each point is one experimental measurement)
plt.scatter(orientations, tuning_noisy, color="coral", s=80, zorder=5,
            edgecolors="white", linewidths=0.5, label="Measured responses")
# zorder=5 means the scatter dots appear in front of the line

plt.xlabel("Stimulus orientation (degrees)")
plt.ylabel("Firing rate (Hz)")
plt.title("Orientation Tuning Curve: Visual Cortex Neuron")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)
plt.xlim(-5, 185)
plt.tight_layout()
plt.show()


---

> ## Going deeper (optional on a first pass)
>
> **Colour maps (`cmap=`).** Matplotlib ships with many colour maps. Good choices
> for sequential data (low to high): `"viridis"` (default, perceptually uniform),
> `"plasma"`, `"magma"`, `"cividis"` (colour-blind safe). For diverging data
> (negative to positive, centred at zero): `"RdBu_r"`, `"coolwarm"`. Avoid `"jet"`
> and `"rainbow"` -- they are not perceptually uniform and create false visual
> features. `"viridis"` is almost always the right default.
>
> **`vmin=` and `vmax=`** in `plt.scatter(c=..., vmin=0, vmax=50)` clamp the
> colour scale to a fixed range, allowing fair comparison between multiple plots.
>
> **Marker styles (`marker=`).** Beyond the default circle `"o"`, scatter accepts
> `"s"` (square), `"^"` (triangle up), `"x"`, `"+"`, `"D"` (diamond), and many
> more. Useful for making multiple groups distinguishable without colour.
>
> **`zorder=`.** Controls drawing order. Higher `zorder` means drawn later (on top).
> Use it when you have scatter points you want to appear above grid lines or lines.
>
> **Regression line.** `np.polyfit(x, y, deg=1)` fits a line (deg=1 is linear).
> Plot it with `plt.plot(x, np.polyval(coefficients, x))` to add a trend line.
>
> **`plt.hexbin(x, y, gridsize=30)`** is a density version of scatter: when you
> have thousands of overlapping points, it bins them into hexagons coloured by count.
> Much cleaner than 10,000 overlapping semi-transparent circles.

---

### Common questions and confusions

**"I passed `c=` an array but all points are the same colour."** The array must be
numeric (not strings) and the same length as `x` and `y`. If it is a list of colour
names, Matplotlib may treat them as something else.

**"The colour bar appeared but it is on the wrong side."** Call `plt.colorbar()` and
pass the scatter object returned by `plt.scatter()`: `cbar = plt.colorbar(scatter)`.
Without that reference, Matplotlib may attach the bar to the wrong object when
multiple things are plotted.

**"Should I use `plt.scatter` or `plt.plot` with a marker style like `'o'`?"**
Use `plt.scatter()` when you need per-point colours or sizes. Use `plt.plot(x, y,
'o')` for a uniform dot plot. `plt.scatter()` is more powerful but slightly slower
for very large datasets.

**"My points all overlap and I cannot see anything."** Decrease `alpha` (try 0.2-0.4
for dense clouds), increase figure size, or switch to `plt.hexbin()` for very large
datasets.

---

### Your exercises

1. In Example 3a, change `s=50` to `s=firing_rate * 3` so that larger dots mean
   higher firing rate. Predict what the plot looks like, then run.
2. Change the colour map in Example 3b from `"plasma"` to `"viridis"`. How does
   the visual pattern change?
3. Add a colour bar to Example 3a by making `c=firing_rate` and adding
   `plt.colorbar()`. Label the bar with `cbar.set_label("Firing rate (Hz)")`.
4. In Example 3c, change `preferred_orientation` to 135 degrees. Where does the
   peak move?
5. Create scatter data: 50 points where `x = np.random.normal(0, 1, 50)` and
   `y = 2 * x + np.random.normal(0, 0.5, 50)`. Plot it and add a title "Linear
   relationship with noise."
6. *(Stretch.)* Create two groups of 60 neurons each. Group 1 fires around 20 Hz
   (mean), Group 2 fires around 40 Hz (mean), both with std=5 Hz. Plot both groups
   as separate scatter calls on the same figure, using different colours and marker
   styles, with a legend identifying each group.

---

### The irreducible core

1. Use **scatter** for independent observations, **line** for ordered sequences.
2. `plt.scatter(x, y, c=array, s=sizes, alpha=)` controls colour, size, and
   transparency per point.
3. When `c=` is a numeric array, add `plt.colorbar(scatter_object)` to show the
   scale.
4. Choose colour maps wisely: `"viridis"` (sequential), `"RdBu_r"` (diverging),
   never `"jet"`.
5. `edgecolors="white"` with a small `linewidths=` makes dense scatter plots much
   easier to read.

**You have got it when:** you can produce a scatter plot of two variables, coloured
by a third variable with a labelled colour bar, and explain to a colleague why you
chose scatter instead of a line.


In [ ]:
# Exercise 1
# Change s=50 to s=firing_rate * 3 in Example 3a -- dots scale with firing rate

# your code here


In [ ]:
# Exercise 2
# Change cmap from 'plasma' to 'viridis' in Example 3b

# your code here


In [ ]:
# Exercise 3
# Add colour bar to Example 3a using c=firing_rate

# your code here


In [ ]:
# Exercise 4
# Change preferred_orientation to 135 degrees in Example 3c

# your code here


In [ ]:
# Exercise 5
# Scatter: x ~ N(0,1), y = 2x + noise; add title

# your code here


In [ ]:
# Exercise 6 (Stretch)
# Two groups of neurons, different colours + markers, legend

# your code here


---
## Item 4 — Histograms

> **By the end of this section you will be able to:** create a histogram with
> `plt.hist()`, choose an appropriate number of bins, overlay two histograms for
> comparison, normalise to a probability density, and interpret what a histogram
> tells you about the distribution of a dataset.

---

### What a histogram actually shows

**Distribution of patient ages in a clinic.** Suppose you record the age of every
patient who visited a clinic in a month. A list of 300 ages is unreadable. But if
you divide the age range into 10-year buckets and count how many patients fall in
each, you can instantly see: are most patients young? Elderly? Spread evenly? That
picture is a histogram.

**Grading distribution in an exam.** Stack 120 exam scores into grade bands (0-10,
10-20, ..., 90-100) and you can immediately see whether the class clustered around
70%, whether there are two groups (the prepared and the unprepared), whether the
distribution is symmetric or skewed. No list of numbers tells you this at a glance.

A **histogram** divides a range of values into equal-width **bins** and draws a bar
whose height is the count (or proportion) of values that fall into each bin. It
visualises the **distribution** -- not individual data points, but the overall shape
of where values concentrate.

### `plt.hist()` syntax

```python
plt.hist(data,
         bins=20,           # number of bins (or a list of bin edges)
         edgecolor="white", # colour of the lines between bars (makes bars distinct)
         density=False)     # if True, y-axis becomes probability density (area=1)
```

**Choosing bins.** Too few bins (e.g. 3) and the shape is hidden. Too many (e.g.
200 for 100 data points) and the bars become sparse and jagged. As a starting rule,
try `bins=int(np.sqrt(len(data)))` (square root of sample size) and adjust by eye.

### Overlapping histograms for comparison

To compare two distributions on the same axes, use `alpha` to make both visible:

```python
plt.hist(data_A, bins=20, alpha=0.6, label="Condition A")
plt.hist(data_B, bins=20, alpha=0.6, label="Condition B")
plt.legend()
```

Without `alpha`, one histogram completely covers the other.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Worked Example 4a: distribution of inter-spike intervals ---
# "Inter-spike interval" (ISI) = the time gap between consecutive spikes (action
# potentials). ISIs tell us about a neuron's firing regularity. A neuron that fires
# at a very steady rate has narrow ISI distribution; a bursty neuron has a wide one.

rng = np.random.default_rng(seed=17)

# Simulate ISIs from an exponential distribution (characteristic of Poisson firing)
# mean_rate = 20 Hz means an average gap of 1/20 s = 50 ms between spikes
mean_rate = 20   # Hz
n_spikes = 400
isis = rng.exponential(scale=1000 / mean_rate, size=n_spikes)  # ISIs in ms

plt.figure(figsize=(8, 5))
plt.hist(isis,
         bins=30,                # 30 bins is reasonable for 400 samples
         color="steelblue",
         edgecolor="white",      # white borders between bars help separate them
         linewidth=0.8)

plt.xlabel("Inter-spike interval (ms)")
plt.ylabel("Count")
plt.title("Distribution of Inter-Spike Intervals (Simulated Poisson Neuron, 20 Hz)")
plt.grid(True, linestyle="--", alpha=0.3, axis="y")   # horizontal grid only
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Worked Example 4b: comparing two conditions with overlapping histograms ---
# Compare firing rates of two neuron populations:
# "Excitatory neurons" (fire frequently) vs "inhibitory neurons" (tend to be quieter)
# These are real neuron subtypes: excitatory neurons activate other neurons,
# inhibitory neurons suppress them. They have different typical firing rates.

rng = np.random.default_rng(seed=33)
n = 200

excitatory_rates = rng.gamma(shape=3, scale=10, size=n)   # right-skewed, mean ~30 Hz
inhibitory_rates = rng.gamma(shape=8, scale=3, size=n)    # narrower, mean ~24 Hz

# --- density=True normalises so that bar areas sum to 1 ---
# This lets us compare shapes fairly even if sample sizes differed
plt.figure(figsize=(9, 5))

plt.hist(excitatory_rates,
         bins=25,
         alpha=0.6,              # make first histogram semi-transparent
         color="tomato",
         edgecolor="white",
         density=True,           # show probability density, not raw count
         label="Excitatory neurons")

plt.hist(inhibitory_rates,
         bins=25,
         alpha=0.6,              # and the second, so both are visible
         color="steelblue",
         edgecolor="white",
         density=True,
         label="Inhibitory neurons")

plt.xlabel("Firing rate (Hz)")
plt.ylabel("Probability density")
plt.title("Firing Rate Distributions: Excitatory vs Inhibitory Neurons")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3, axis="y")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Worked Example 4c: effect of bin count on histogram appearance ---
# Same data, four different bin counts -- see how the picture changes

rng = np.random.default_rng(seed=9)
data = rng.normal(loc=50, scale=10, size=500)   # 500 values, mean=50, std=10

fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=False)
bin_choices = [5, 15, 30, 80]

for ax, n_bins in zip(axes, bin_choices):
    ax.hist(data, bins=n_bins, color="mediumseagreen", edgecolor="white", linewidth=0.5)
    ax.set_title(f"bins={n_bins}")
    ax.set_xlabel("Value")
    ax.set_ylabel("Count" if n_bins == 5 else "")

fig.suptitle("How Bin Count Changes the Histogram's Message", fontsize=13)
plt.tight_layout()
plt.show()


---

> ## Going deeper (optional on a first pass)
>
> **`plt.hist()` return values.** The function returns a tuple `(counts, bin_edges,
> patches)`. `counts` is a NumPy array of bar heights; `bin_edges` has one more
> element than `counts` (the left and right edges of every bin). You rarely need
> these for display but they are essential if you want to, for example, find the
> mode bin or draw a custom curve on top.
>
> **Manual bin edges.** Instead of a bin count, you can pass an explicit array of
> edges: `plt.hist(data, bins=np.arange(0, 200, 10))` makes bins from 0-10, 10-20,
> ..., 190-200. Use this when the bin width has a natural unit (every 10 ms, every
> 5 Hz) rather than letting Matplotlib pick.
>
> **`density=True` vs `weights=`.** `density=True` normalises the entire histogram
> so the bar areas integrate to 1 (making it a probability density estimate). An
> alternative is `weights=np.ones(len(data)) / len(data)`, which normalises to
> proportions (bar heights sum to 1 but are not a density).
>
> **Kernel density estimation (KDE).** A histogram is a rough approximation of the
> underlying probability distribution. A KDE is a smooth version using overlapping
> kernel functions. `scipy.stats.gaussian_kde` produces a KDE that can be plotted as
> a smooth curve on top of a histogram.
>
> **Log scale.** `plt.yscale("log")` changes the y-axis to logarithmic, useful when
> counts span several orders of magnitude (a few very long ISIs alongside thousands
> of short ones). `plt.xscale("log")` does the same for x.
>
> **Cumulative histogram.** `plt.hist(data, cumulative=True, density=True)` produces
> the empirical cumulative distribution function (CDF): for each x value, the bar
> height tells you what fraction of data falls at or below x.

---

### Common questions and confusions

**"My two histograms overlap and I can only see one."** Add `alpha=0.6` (or similar)
to both `plt.hist()` calls. Without transparency, the second histogram paints over
the first.

**"Should I use `bins=` as a count or provide edges?"** For exploration, use a
count (start with 20-30). For presentation or when bins need meaningful widths,
provide explicit edges.

**"What is the difference between `density=True` and just dividing counts by N?"**
`density=True` divides by both N and the bin width, so that the bar areas (not
heights) integrate to 1. Dividing counts by N gives proportions per bin. The
choice depends on whether you want to compare heights directly or overlay a
continuous probability density function.

**"Why does my histogram have a gap on one side?"** By default, `plt.hist()`
includes the left edge but excludes the right edge of each bin. The last bin
includes both edges. Gaps usually mean some values fall outside the plotted range.

---

### Your exercises

1. In Example 4a, change `bins=30` to `bins=5`. What information is lost?
2. Change `density=False` to `density=True` in Example 4a. What changes on the
   y-axis?
3. In Example 4b, make the excitatory histogram solid (remove `alpha`) and re-run.
   Why is this a problem? Restore `alpha=0.6`.
4. Generate 1000 samples from `rng.normal(0, 1, 1000)` and plot a histogram with
   `density=True`. Then overlay a smooth normal curve using
   `x = np.linspace(-4, 4, 200)` and
   `y = np.exp(-x**2 / 2) / np.sqrt(2 * np.pi)` with `plt.plot(x, y)`.
5. Create two groups: `control = rng.normal(50, 8, 150)` and
   `drug = rng.normal(60, 10, 150)`. Plot overlapping histograms comparing them
   with proper labels, title, and legend.
6. *(Stretch.)* Use the `plt.hist()` return values `(counts, edges, _)` to find
   and print the bin with the highest count, and draw a vertical line at that
   bin's centre with `plt.axvline()`.

---

### The irreducible core

1. A histogram shows the **distribution** of values: the shape, centre, spread,
   and any skew.
2. `plt.hist(data, bins=N, edgecolor="white")` is the basic call; choose N by eye
   after starting near the square root of your sample size.
3. `density=True` normalises the area to 1, enabling comparison between differently
   sized samples or overlay with probability densities.
4. For comparing two distributions, overlay with `alpha=0.6` on both, use distinct
   colours, and add a legend.
5. Too few bins hide the shape; too many create spurious noise. Bin choice is a
   judgment call, not a formula.

**You have got it when:** you can plot a distribution of neural inter-spike intervals,
choose a sensible number of bins, overlay a second condition with transparency, and
read off from the figure which condition has a wider spread.


In [ ]:
# Exercise 1
# Change bins=30 to bins=5 in Example 4a -- what is lost?

# your code here


In [ ]:
# Exercise 2
# Change density=False to density=True in Example 4a

# your code here


In [ ]:
# Exercise 3
# Make excitatory histogram solid (no alpha) then restore alpha=0.6

# your code here


In [ ]:
# Exercise 4
# Histogram of N(0,1) with density=True; overlay the true normal curve

# your code here


In [ ]:
# Exercise 5
# Overlapping histograms: control vs drug condition

# your code here


In [ ]:
# Exercise 6 (Stretch)
# Use plt.hist() return values to find the modal bin, then draw axvline at its centre

# your code here


---
## Item 5 — Subplots

> **By the end of this section you will be able to:** create multi-panel figures
> with `plt.subplots()`, address individual axes to draw different plots in each
> panel, prevent label overlap with `plt.tight_layout()`, and share axes between
> panels for aligned comparison.

---

### Why multiple panels?

**A medical chart with multiple panels.** A patient's chart might show blood
pressure, heart rate, and blood oxygen on three stacked panels, all aligned in
time. You glance down the chart and the temporal alignment is immediate: the blood
pressure spike at minute 4 corresponds to the oxygen dip at minute 4. Putting them
on one set of axes with three different y-axes would be illegible.

**A newspaper with columns of graphs.** A data journalist presenting election
results for four regions puts four bar charts side by side so the reader can compare
across regions at the same scale and in one glance. One giant combined chart would
be confusing; four tiny charts scattered across the page lose the comparison.

In neuroscience, the most common use is: voltage trace on top, raster plot below,
both aligned on the same time axis so the relationship between voltage and spiking
is visible.

*Raster plot:* A raster plot shows spike times as vertical tick marks across many
trials. Each row is one trial; each tick is one spike. It is the standard way to
visualise many trials of neural data at once.

### `plt.subplots()` syntax

```python
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(10, 6))
```

This returns:
- `fig`: the whole figure object (canvas).
- `axes`: an array of individual axes objects.

For a 2x1 layout, `axes` is a 1D array: `axes[0]` is the top panel, `axes[1]` is
the bottom.
For a 2x3 layout, `axes` is a 2D array: `axes[0, 0]` is top-left, `axes[1, 2]` is
bottom-right.

### Addressing individual axes

Once you have the axes objects, you call plotting methods on them directly:

```python
axes[0].plot(t, voltage)           # draw in the top panel
axes[0].set_ylabel("Voltage (mV)") # label the top panel's y-axis
axes[1].scatter(t_spikes, trial_ids, marker="|")
axes[1].set_ylabel("Trial")
axes[1].set_xlabel("Time (ms)")    # label only the bottom panel's x-axis
```

Note: when using the OO interface, `plt.xlabel()` becomes `ax.set_xlabel()`, and
`plt.title()` becomes `ax.set_title()`. The pattern is: method names gain `set_`.

### Sharing axes

```python
fig, axes = plt.subplots(2, 1, sharex=True)   # both panels share the same x-axis
```

With `sharex=True`, zooming in one panel automatically zooms the other. Crucially,
only the bottom panel needs an x-axis label; the top one's ticks are suppressed.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Worked Example 5a: voltage trace + raster plot, shared time axis ---

rng = np.random.default_rng(seed=44)
t = np.linspace(0, 200, 2000)          # 0 to 200 ms

# --- Panel 1: membrane voltage with two spikes ---
voltage = -70 + rng.normal(0, 1.5, size=len(t))
for spike_time in [60, 130]:
    mask = (t >= spike_time) & (t <= spike_time + 2)
    voltage[mask] = 40 * np.exp(-((t[mask] - spike_time - 0.5) ** 2) / 0.2)

# --- Panel 2: raster plot across 20 trials ---
# Each trial is one row of tick marks; spikes occur near t=60 and t=130 ms with jitter
n_trials = 20
all_spike_times = []
all_trial_ids = []
for trial in range(n_trials):
    for mean_spike in [60, 130]:
        if rng.random() > 0.15:     # 85% probability of spiking on any trial
            spike = mean_spike + rng.normal(0, 4)   # jitter: +-4 ms variation
            all_spike_times.append(spike)
            all_trial_ids.append(trial + 1)

# Create the figure: 2 rows, 1 column, sharing the x-axis
fig, axes = plt.subplots(nrows=2, ncols=1,
                         figsize=(11, 6),
                         sharex=True)           # sharex links the two time axes

# --- Top panel: voltage trace ---
axes[0].plot(t, voltage, color="steelblue", linewidth=1.2)
axes[0].axhline(y=-55, color="crimson", linestyle="--", linewidth=1,
                label="Threshold")
axes[0].set_ylabel("Voltage (mV)")
axes[0].set_title("Voltage Trace and Spike Raster (20 Trials)")
axes[0].set_ylim(-85, 55)
axes[0].legend(loc="upper right", fontsize=9)
axes[0].grid(True, linestyle="--", alpha=0.25)

# --- Bottom panel: raster plot ---
# marker="|" draws vertical tick marks, perfect for spike rasters
axes[1].scatter(all_spike_times, all_trial_ids,
                marker="|",        # vertical tick for each spike
                s=80,
                color="black",
                linewidths=1.5)
axes[1].set_ylabel("Trial number")
axes[1].set_xlabel("Time (ms)")    # only the bottom panel needs this
axes[1].set_ylim(0, n_trials + 1)
axes[1].set_xlim(0, 200)

plt.tight_layout()      # prevent panels from overlapping each other
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Worked Example 5b: 2x2 grid comparing four neurons ---
# Each panel shows the voltage trace of a different simulated neuron

rng = np.random.default_rng(seed=55)
t = np.linspace(0, 150, 1500)

neuron_configs = [
    {"name": "Neuron A (fast)", "spikes": [30, 60, 90, 120], "noise": 1.5, "color": "steelblue"},
    {"name": "Neuron B (slow)", "spikes": [75],               "noise": 2.0, "color": "coral"},
    {"name": "Neuron C (bursting)", "spikes": [40, 43, 46, 110, 113], "noise": 1.0, "color": "mediumseagreen"},
    {"name": "Neuron D (silent)",   "spikes": [],               "noise": 3.0, "color": "goldenrod"},
]

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 7),
                         sharex=True, sharey=True)   # share both axes for fair comparison
axes_flat = axes.flatten()    # convert 2D array to 1D for easy looping

for ax, cfg in zip(axes_flat, neuron_configs):
    # Build the voltage trace for this neuron
    v = -70 + rng.normal(0, cfg["noise"], size=len(t))
    for spike_time in cfg["spikes"]:
        mask = (t >= spike_time) & (t <= spike_time + 2)
        v[mask] = 40 * np.exp(-((t[mask] - spike_time - 0.5) ** 2) / 0.2)

    ax.plot(t, v, color=cfg["color"], linewidth=1.2)
    ax.axhline(y=-55, color="grey", linestyle=":", linewidth=1, alpha=0.7)
    ax.set_title(cfg["name"])
    ax.set_ylim(-85, 55)
    ax.grid(True, linestyle="--", alpha=0.2)

# Add shared axis labels by targeting the outer axes
for ax in axes[1, :]:         # bottom row: add x-axis label
    ax.set_xlabel("Time (ms)")
for ax in axes[:, 0]:         # left column: add y-axis label
    ax.set_ylabel("Voltage (mV)")

fig.suptitle("Four Simulated Neurons: Voltage Traces", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Worked Example 5c: stimulus and response in aligned panels ---
# A classic neuroscience experiment: show a stimulus, record the neural response.
# Two panels aligned on the same time axis show the relationship clearly.

rng = np.random.default_rng(seed=66)
t = np.linspace(0, 300, 3000)   # 0 to 300 ms

# --- Stimulus: a square pulse from t=50 to t=150 ms ---
stimulus = np.zeros(len(t))
stimulus[(t >= 50) & (t <= 150)] = 1.0    # 1 = stimulus on, 0 = off

# --- Neural response: firing rate ramps up during stimulus, decays after ---
response_rate = np.zeros(len(t))
response_rate[(t >= 50) & (t <= 150)] = (
    25 * (1 - np.exp(-(t[(t >= 50) & (t <= 150)] - 50) / 20))
)
# Decay after stimulus offset
offset_mask = t > 150
response_rate[offset_mask] = 25 * np.exp(-(t[offset_mask] - 150) / 40)
response_rate += rng.normal(0, 1, len(t))
response_rate = np.clip(response_rate, 0, None)

# --- Two panels, shared x-axis ---
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True,
                         gridspec_kw={"height_ratios": [1, 2.5]})
# gridspec_kw lets you control relative heights: stimulus panel is 1/3.5 of the total

# Top panel: stimulus
axes[0].fill_between(t, stimulus, color="orange", alpha=0.7, label="Stimulus")
axes[0].set_ylabel("Stimulus
(on/off)")
axes[0].set_ylim(-0.1, 1.4)
axes[0].set_yticks([0, 1])
axes[0].set_yticklabels(["off", "on"])
axes[0].set_title("Stimulus and Neural Response")
axes[0].legend(loc="upper right")

# Bottom panel: firing rate
axes[1].plot(t, response_rate, color="steelblue", linewidth=1.5, label="Firing rate")
axes[1].axvline(x=50,  color="orange", linestyle="--", alpha=0.7, label="Stimulus on")
axes[1].axvline(x=150, color="red",    linestyle="--", alpha=0.7, label="Stimulus off")
axes[1].set_ylabel("Firing rate (Hz)")
axes[1].set_xlabel("Time (ms)")
axes[1].set_xlim(0, 300)
axes[1].legend(loc="upper right")
axes[1].grid(True, linestyle="--", alpha=0.25)

plt.tight_layout()
plt.show()


---

> ## Going deeper (optional on a first pass)
>
> **`gridspec_kw={"height_ratios": [1, 3]}`** (shown in Example 5c) lets you
> set different heights for each row. Similarly `width_ratios` controls column
> widths. `[1, 3]` makes the second row three times taller than the first.
>
> **`plt.GridSpec`** for irregular layouts. When `plt.subplots()` is not flexible
> enough (e.g. one large left panel and two small stacked right panels), use
> `matplotlib.gridspec.GridSpec` directly. It allows axes to span multiple rows
> or columns.
>
> **`axes.flatten()`** converts a 2D NumPy array of axes (returned by a multi-row,
> multi-column `plt.subplots()`) into a 1D array so you can loop over it simply.
> Alternatively, create with `plt.subplots(2, 3, squeeze=False)` to always get a
> 2D array regardless of dimensions.
>
> **`fig.savefig("name.pdf", bbox_inches="tight")`** is the publication standard:
> PDF is vector format (infinitely scalable), and `bbox_inches="tight"` trims the
> excess white space that `tight_layout()` does not always catch.
>
> **`plt.subplots_adjust(hspace=0.4, wspace=0.3)`** manually controls the vertical
> (`hspace`) and horizontal (`wspace`) spacing between panels, as fractions of the
> average axis height/width. Use when `tight_layout()` is not enough.
>
> **`fig.add_subplot()`** adds a single new axes to an existing figure at an
> arbitrary position. Useful for building up figures incrementally. Rarely needed
> when `plt.subplots()` handles the whole layout upfront.
>
> **Inset axes.** `fig.add_axes([left, bottom, width, height])` (all in figure
> fractions 0-1) places a new axes anywhere inside the figure, even overlapping
> another. This is how people add a zoomed-in inset panel.

---

### Common questions and confusions

**"I called `plt.xlabel()` but the label only appeared on the bottom-right panel."**
When using the OO interface (`fig, axes = plt.subplots()`), use
`ax.set_xlabel("...")` on the specific axes object. `plt.xlabel()` applies to
whatever the current axes happens to be.

**"My panels overlap even after `plt.tight_layout()`."** Add
`plt.subplots_adjust(hspace=0.5)` or increase `figsize`. Long titles are the most
common culprit; shorten them or move them to `fig.suptitle()`.

**"With `sharex=True`, I cannot zoom one panel independently."** That is by design.
`sharex` means the panels are locked together. If you need independent zooming,
do not use `sharex`. You can still align them visually by calling `ax.set_xlim()`
with identical limits on each.

**"How do I remove empty panels in a grid that is not evenly filled?"** Use
`ax.set_visible(False)` on the axes objects you do not want, or use
`fig.delaxes(ax)` to remove them entirely.

**"The shared x-axis still shows tick labels on the top panels."** With
`sharex=True`, Matplotlib suppresses top-panel tick labels automatically. If it
does not, call `plt.setp(axes[0].get_xticklabels(), visible=False)`.

---

### Your exercises

1. In Example 5a, change `sharex=True` to `sharex=False`. What changes about the
   two panels? Which version is more useful for reading the data?
2. In Example 5b, add a title to the whole figure using `fig.suptitle("My Title",
   fontsize=14)`. Where does it appear?
3. Create a 1x3 subplot grid. In the three panels, plot `sin(x)`, `cos(x)`, and
   `sin(x) * cos(x)` for x from 0 to 4*pi. Add a title to each panel.
4. Take Example 5c and change `height_ratios` to `[1, 1]` so both panels are equal
   height. Is it still readable?
5. In Example 5a, add a second threshold line at y=-65 mV to the voltage panel
   using `axes[0].axhline()`, labelled "Near-threshold depolarisation".
6. *(Stretch.)* Create a 2x2 grid. Top-left: line plot. Top-right: scatter plot.
   Bottom-left: histogram. Bottom-right: an empty axes that shows only a text
   label in the centre using `ax.text(0.5, 0.5, "Summary goes here",
   ha="center", va="center", transform=ax.transAxes, fontsize=14)`. Add a shared
   figure title. This is the structure of a typical paper figure.

---

### The irreducible core

1. `fig, axes = plt.subplots(nrows, ncols, figsize=)` creates the figure and all
   axes at once.
2. Address individual panels via `axes[i]` (1D grid) or `axes[i, j]` (2D grid).
3. Use `ax.set_xlabel()`, `ax.set_ylabel()`, `ax.set_title()` on each axes object
   (not `plt.xlabel()`) when in multi-panel mode.
4. `sharex=True` or `sharey=True` locks axes together for aligned comparison.
5. Always call `plt.tight_layout()` before showing or saving a multi-panel figure.

**You have got it when:** you can build a two-panel figure with a voltage trace on
top and a raster plot on the bottom, sharing a time axis, with proper labels on
each panel, without any text clipping, and explain to a colleague what `sharex=True`
does and why it matters.


In [ ]:
# Exercise 1
# In Example 5a, change sharex=True to sharex=False -- what changes?

# your code here


In [ ]:
# Exercise 2
# Add fig.suptitle() to Example 5b

# your code here


In [ ]:
# Exercise 3
# 1x3 grid: sin(x), cos(x), sin(x)*cos(x) for x in [0, 4*pi]

# your code here


In [ ]:
# Exercise 4
# Change height_ratios to [1, 1] in Example 5c -- is it still readable?

# your code here


In [ ]:
# Exercise 5
# Add second threshold line at y=-65 mV to the voltage panel in Example 5a

# your code here


In [ ]:
# Exercise 6 (Stretch)
# 2x2 grid: line, scatter, histogram, text summary panel

# your code here


---
---

# Solutions -- try first!

Work through every exercise yourself before scrolling here. The struggle is where
the learning happens. These solutions are commented to explain the choices made,
not just the mechanics.


## Item 1 Solutions — Your First Plot

In [ ]:
# Solution 1-1: x = [1..5], y = 2x; expect a straight line through the origin
import matplotlib.pyplot as plt
x = [1, 2, 3, 4, 5]
y = [2, 4, 6, 8, 10]
plt.figure(figsize=(6, 4))
plt.plot(x, y)
plt.title("y = 2x (straight line)")
plt.xlabel("x")
plt.ylabel("y")
plt.show()


In [ ]:
# Solution 1-2: sine wave
import matplotlib.pyplot as plt
import numpy as np
x = np.linspace(0, 2 * np.pi, 200)
plt.figure(figsize=(8, 4))
plt.plot(x, np.sin(x))
plt.title("sin(x) from 0 to 2*pi")
plt.xlabel("x (radians)")
plt.ylabel("sin(x)")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Solution 1-3: two plt.plot() calls on the same axes
import matplotlib.pyplot as plt
import numpy as np
x = np.linspace(0, 2 * np.pi, 200)
plt.figure()
plt.plot(x, np.sin(x), label="sin")
plt.plot(x, np.cos(x), label="cos")   # added to the same axes automatically
plt.legend()
plt.title("Both lines appear on the same axes (no plt.figure() in between)")
plt.show()


In [ ]:
# Solution 1-4: portrait figure -- the trace looks stretched vertically
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=42)
t = np.linspace(0, 100, 1000)
voltage = -70 + rng.normal(0, 1.5, size=len(t))
for spike_time in [20, 50, 80]:
    mask = (t >= spike_time) & (t <= spike_time + 2)
    voltage[mask] = 40 * np.exp(-((t[mask] - spike_time - 0.5) ** 2) / 0.2)
plt.figure(figsize=(3, 6))   # portrait: narrow and tall
plt.plot(t, voltage)
plt.title("Portrait figure")
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (mV)")
plt.show()


In [ ]:
# Solution 1-5: add third spike at t=65 ms
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=42)
t = np.linspace(0, 100, 1000)
voltage = -70 + rng.normal(0, 1.5, size=len(t))
for spike_time in [20, 50, 65, 80]:   # added 65 here
    mask = (t >= spike_time) & (t <= spike_time + 2)
    voltage[mask] = 40 * np.exp(-((t[mask] - spike_time - 0.5) ** 2) / 0.2)
plt.figure(figsize=(10, 4))
plt.plot(t, voltage, color="steelblue")
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (mV)")
plt.title("Voltage trace: four spikes (added at 65 ms)")
plt.show()


In [ ]:
# Solution 1-6 (Stretch): sin(x) and sin(2x), same axes then separate figures
import matplotlib.pyplot as plt
import numpy as np
x = np.linspace(0, 2 * np.pi, 300)

# Same axes (no plt.figure() in between):
plt.figure(figsize=(8, 4))
plt.plot(x, np.sin(x), label="sin(x)")
plt.plot(x, np.sin(2 * x), label="sin(2x)")   # second call lands on same axes
plt.legend()
plt.title("sin(x) and sin(2x) on the SAME axes")
plt.show()

# Separate figures (plt.figure() before each):
plt.figure(figsize=(6, 3))
plt.plot(x, np.sin(x), color="steelblue")
plt.title("sin(x) -- separate figure 1")
plt.show()

plt.figure(figsize=(6, 3))
plt.plot(x, np.sin(2 * x), color="coral")
plt.title("sin(2x) -- separate figure 2")
plt.show()


## Item 2 Solutions — Customising Plots

In [ ]:
# Solution 2-1: change Neuron A's colour and Neuron B's linestyle
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=0)
t = np.linspace(0, 100, 1000)
voltage_A = -70 + rng.normal(0, 1.5, size=len(t))
voltage_B = -70 + rng.normal(0, 3.0, size=len(t))
for spike_time in [40, 75]:
    mask = (t >= spike_time) & (t <= spike_time + 2)
    voltage_A[mask] = 40 * np.exp(-((t[mask] - spike_time - 0.5) ** 2) / 0.2)
mask_B = (t >= 60) & (t <= 62)
voltage_B[mask_B] = 35 * np.exp(-((t[mask_B] - 60.5) ** 2) / 0.2)

plt.figure(figsize=(11, 4))
plt.plot(t, voltage_A, color="#E63946", linewidth=1.8, linestyle="-",  label="Neuron A")
plt.plot(t, voltage_B, color="coral",   linewidth=1.8, linestyle="-.", label="Neuron B")
plt.axhline(y=-55, color="crimson", linestyle=":", linewidth=1.5, label="Threshold")
plt.xlabel("Time (ms)")
plt.ylabel("Membrane voltage (mV)")
plt.title("Exercise 2-1: updated colours and linestyle")
plt.legend()
plt.xlim(0, 100)
plt.ylim(-85, 55)
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Solution 2-2: add a vertical line at x=50 ms
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=0)
t = np.linspace(0, 100, 1000)
voltage_A = -70 + rng.normal(0, 1.5, size=len(t))
for spike_time in [40, 75]:
    mask = (t >= spike_time) & (t <= spike_time + 2)
    voltage_A[mask] = 40 * np.exp(-((t[mask] - spike_time - 0.5) ** 2) / 0.2)

plt.figure(figsize=(11, 4))
plt.plot(t, voltage_A, color="steelblue", label="Neuron A")
plt.axvline(x=50, color="green", linestyle="--", linewidth=1.5, label="Midpoint (50 ms)")
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (mV)")
plt.title("Vertical line marking the midpoint in time")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Solution 2-3: set ylim(-100, 60) -- adds more headroom below resting potential
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=0)
t = np.linspace(0, 100, 1000)
voltage_A = -70 + rng.normal(0, 1.5, size=len(t))
for spike_time in [40, 75]:
    mask = (t >= spike_time) & (t <= spike_time + 2)
    voltage_A[mask] = 40 * np.exp(-((t[mask] - spike_time - 0.5) ** 2) / 0.2)

plt.figure(figsize=(11, 4))
plt.plot(t, voltage_A, color="steelblue")
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (mV)")
plt.title("ylim(-100, 60): more empty space at the bottom")
plt.ylim(-100, 60)    # more space below the resting potential
plt.tight_layout()
plt.show()


In [ ]:
# Solution 2-4: alpha=0.8 makes the grid more prominent -- competes with data
import matplotlib.pyplot as plt
import numpy as np
x = np.linspace(0, 10, 100)
plt.figure(figsize=(7, 4))
plt.plot(x, np.sin(x), linewidth=2)
plt.grid(True, linestyle="--", alpha=0.8)   # heavy grid: notice how it distracts
plt.title("Heavy grid (alpha=0.8): data vs grid compete for attention")
plt.xlabel("x")
plt.ylabel("sin(x)")
plt.tight_layout()
plt.show()


In [ ]:
# Solution 2-5: label the baseline line; move legend to upper right
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=3)
time_points = np.linspace(0, 500, 50)
rate_low  = 5 + 10 * np.exp(-time_points / 150) + rng.normal(0, 0.8, 50)
rate_high = 5 + 30 * np.exp(-time_points / 150) + rng.normal(0, 1.5, 50)

plt.figure(figsize=(9, 4))
plt.plot(time_points, rate_low,  color="#4C72B0", linewidth=2, label="Low stimulus")
plt.plot(time_points, rate_high, color="#DD8452", linewidth=2, label="High stimulus")
plt.axhline(y=5, color="grey", linestyle=":", linewidth=1.2, label="Baseline (5 Hz)")
plt.xlabel("Time after stimulus onset (ms)")
plt.ylabel("Firing rate (Hz)")
plt.title("Firing Rate: Low vs High Stimulus, legend upper right")
plt.legend(loc="upper right")   # explicit placement
plt.tight_layout()
plt.show()


In [ ]:
# Solution 2-6 (Stretch): three sine waves, full presentation formatting
import matplotlib.pyplot as plt
import numpy as np
t = np.linspace(0, 1, 500)   # 1 second of time

plt.figure(figsize=(10, 5))
plt.plot(t, np.sin(2 * np.pi * 1 * t), color="steelblue",      linestyle="-",
         linewidth=2, label="1 Hz")
plt.plot(t, np.sin(2 * np.pi * 2 * t), color="coral",          linestyle="--",
         linewidth=2, label="2 Hz")
plt.plot(t, np.sin(2 * np.pi * 3 * t), color="mediumseagreen", linestyle="-.",
         linewidth=2, label="3 Hz")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Three Sine Waves at Different Frequencies")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()


## Item 3 Solutions — Scatter Plots

In [ ]:
# Solution 3-1: marker size scales with firing rate
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=11)
n_neurons = 80
stimulus_intensity = rng.uniform(0, 100, n_neurons)
firing_rate = 2 + 0.45 * stimulus_intensity + rng.normal(0, 8, n_neurons)
firing_rate = np.clip(firing_rate, 0, None)

plt.figure(figsize=(7, 5))
plt.scatter(stimulus_intensity, firing_rate,
            color="steelblue",
            s=firing_rate * 3,   # larger dot = higher firing rate
            alpha=0.7,
            edgecolors="white", linewidths=0.5)
plt.xlabel("Stimulus intensity (%)")
plt.ylabel("Firing rate (Hz)")
plt.title("Firing Rate vs Stimulus: dot size scales with rate")
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Solution 3-2: change cmap to 'viridis' in spatial plot
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=22)
n_neurons = 120
x_pos = rng.uniform(0, 1, n_neurons)
y_pos = rng.uniform(0, 1, n_neurons)
firing_rate = 20 * np.exp(-((x_pos - 0.5)**2 + (y_pos - 0.5)**2) / 0.08)
firing_rate += rng.normal(0, 1, n_neurons)
firing_rate = np.clip(firing_rate, 0, None)

plt.figure(figsize=(7, 6))
sc = plt.scatter(x_pos, y_pos, c=firing_rate, s=70, alpha=0.85,
                 cmap="viridis",   # changed from 'plasma'
                 edgecolors="grey", linewidths=0.3)
cbar = plt.colorbar(sc)
cbar.set_label("Firing rate (Hz)")
plt.xlabel("x position (mm)")
plt.ylabel("y position (mm)")
plt.title("viridis colour map: dark=low, yellow=high")
plt.tight_layout()
plt.show()


In [ ]:
# Solution 3-3: add colour bar to firing rate vs stimulus scatter
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=11)
n_neurons = 80
stimulus_intensity = rng.uniform(0, 100, n_neurons)
firing_rate = 2 + 0.45 * stimulus_intensity + rng.normal(0, 8, n_neurons)
firing_rate = np.clip(firing_rate, 0, None)

plt.figure(figsize=(8, 5))
sc = plt.scatter(stimulus_intensity, firing_rate,
                 c=firing_rate,       # colour mapped to firing rate
                 s=60, alpha=0.8,
                 cmap="viridis",
                 edgecolors="white", linewidths=0.5)
cbar = plt.colorbar(sc)
cbar.set_label("Firing rate (Hz)")
plt.xlabel("Stimulus intensity (%)")
plt.ylabel("Firing rate (Hz)")
plt.title("Firing rate vs stimulus: colour bar added")
plt.tight_layout()
plt.show()


In [ ]:
# Solution 3-4: move preferred orientation to 135 degrees
import matplotlib.pyplot as plt
import numpy as np
orientations = np.arange(0, 181, 15)
preferred_orientation = 135    # peak now at 135 degrees
tuning = 40 * np.exp(-((orientations - preferred_orientation)**2) / (2 * 30**2)) + 5
rng = np.random.default_rng(seed=5)
tuning_noisy = tuning + rng.normal(0, 2, len(orientations))

plt.figure(figsize=(8, 5))
plt.plot(orientations, tuning, color="steelblue", linewidth=2, linestyle="--",
         label="True tuning", alpha=0.6)
plt.scatter(orientations, tuning_noisy, color="coral", s=80, zorder=5,
            edgecolors="white", linewidths=0.5, label="Measured")
plt.xlabel("Orientation (degrees)")
plt.ylabel("Firing rate (Hz)")
plt.title("Tuning curve: peak shifted to 135 degrees")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Solution 3-5: scatter of a linear relationship with noise
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=42)
x = rng.normal(0, 1, 50)
y = 2 * x + rng.normal(0, 0.5, 50)

plt.figure(figsize=(6, 5))
plt.scatter(x, y, color="steelblue", s=50, alpha=0.8, edgecolors="white")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Linear relationship with noise")
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Solution 3-6 (Stretch): two neuron groups, different colour and marker
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=99)
n = 60

group1_x = rng.uniform(0, 1, n)
group1_rate = rng.normal(20, 5, n)

group2_x = rng.uniform(0, 1, n)
group2_rate = rng.normal(40, 5, n)

plt.figure(figsize=(8, 5))
plt.scatter(group1_x, group1_rate, color="steelblue", marker="o",
            s=60, alpha=0.7, edgecolors="white", label="Group 1 (~20 Hz)")
plt.scatter(group2_x, group2_rate, color="coral", marker="^",
            s=70, alpha=0.7, edgecolors="white", label="Group 2 (~40 Hz)")
plt.xlabel("Neuron position (mm)")
plt.ylabel("Firing rate (Hz)")
plt.title("Two Neuron Populations: Different Mean Firing Rates")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()


## Item 4 Solutions — Histograms

In [ ]:
# Solution 4-1: bins=5 -- shape of the ISI distribution is obscured
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=17)
isis = rng.exponential(scale=1000 / 20, size=400)

plt.figure(figsize=(7, 4))
plt.hist(isis, bins=5, color="steelblue", edgecolor="white")
plt.xlabel("ISI (ms)")
plt.ylabel("Count")
plt.title("ISI histogram with only 5 bins: shape is hidden")
plt.tight_layout()
plt.show()


In [ ]:
# Solution 4-2: density=True changes y-axis to probability density
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=17)
isis = rng.exponential(scale=1000 / 20, size=400)

plt.figure(figsize=(7, 4))
plt.hist(isis, bins=30, color="steelblue", edgecolor="white", density=True)
plt.xlabel("ISI (ms)")
plt.ylabel("Probability density")  # note the changed label
plt.title("ISI histogram with density=True: y-axis is now probability density")
plt.tight_layout()
plt.show()


In [ ]:
# Solution 4-3: solid excitatory histogram covers the inhibitory one
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=33)
n = 200
excitatory_rates = rng.gamma(shape=3, scale=10, size=n)
inhibitory_rates = rng.gamma(shape=8, scale=3,  size=n)

# Without alpha: completely opaque, one histogram hides the other
plt.figure(figsize=(9, 4))
plt.hist(excitatory_rates, bins=25, color="tomato",    edgecolor="white",
         density=True, label="Excitatory (solid, blocks view)")
plt.hist(inhibitory_rates, bins=25, color="steelblue", edgecolor="white",
         density=True, label="Inhibitory (hidden behind)")
plt.legend()
plt.title("Problem: no alpha -- one histogram blocks the other")
plt.tight_layout()
plt.show()

# With alpha=0.6: both visible, comparison is clear
plt.figure(figsize=(9, 4))
plt.hist(excitatory_rates, bins=25, color="tomato",    edgecolor="white",
         density=True, alpha=0.6, label="Excitatory")
plt.hist(inhibitory_rates, bins=25, color="steelblue", edgecolor="white",
         density=True, alpha=0.6, label="Inhibitory")
plt.legend()
plt.xlabel("Firing rate (Hz)")
plt.ylabel("Probability density")
plt.title("Fixed: alpha=0.6 makes both distributions visible")
plt.tight_layout()
plt.show()


In [ ]:
# Solution 4-4: histogram + overlaid true normal curve
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=1)
data = rng.normal(0, 1, 1000)
x_curve = np.linspace(-4, 4, 200)
y_curve = np.exp(-x_curve**2 / 2) / np.sqrt(2 * np.pi)

plt.figure(figsize=(8, 5))
plt.hist(data, bins=30, color="steelblue", edgecolor="white",
         density=True, alpha=0.7, label="Sampled data")
plt.plot(x_curve, y_curve, color="crimson", linewidth=2.5,
         label="True N(0,1) density")
plt.xlabel("Value")
plt.ylabel("Probability density")
plt.title("1000 Samples from N(0,1) with True Density Overlaid")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Solution 4-5: control vs drug condition comparison
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=7)
control = rng.normal(50, 8, 150)
drug    = rng.normal(60, 10, 150)

plt.figure(figsize=(8, 5))
plt.hist(control, bins=20, alpha=0.6, color="steelblue", edgecolor="white",
         label="Control (mean=50)")
plt.hist(drug,    bins=20, alpha=0.6, color="tomato",    edgecolor="white",
         label="Drug condition (mean=60)")
plt.xlabel("Response score")
plt.ylabel("Count")
plt.title("Control vs Drug Condition: Response Distribution")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3, axis="y")
plt.tight_layout()
plt.show()


In [ ]:
# Solution 4-6 (Stretch): find the modal bin using plt.hist() return values
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=17)
isis = rng.exponential(scale=50, size=400)

plt.figure(figsize=(8, 5))
counts, edges, patches = plt.hist(isis, bins=30, color="steelblue", edgecolor="white")

# Find the bin with the highest count
modal_bin_index = np.argmax(counts)              # index of the tallest bar
modal_bin_centre = (edges[modal_bin_index] + edges[modal_bin_index + 1]) / 2

print(f"Modal bin index: {modal_bin_index}")
print(f"Modal bin centre: {modal_bin_centre:.1f} ms")
print(f"Count in modal bin: {counts[modal_bin_index]:.0f}")

plt.axvline(x=modal_bin_centre, color="crimson", linestyle="--", linewidth=2,
            label=f"Modal bin centre ({modal_bin_centre:.1f} ms)")
plt.xlabel("ISI (ms)")
plt.ylabel("Count")
plt.title("ISI Histogram: Modal Bin Marked")
plt.legend()
plt.tight_layout()
plt.show()


## Item 5 Solutions — Subplots

In [ ]:
# Solution 5-1: sharex=False -- panels are now independent
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=44)
t = np.linspace(0, 200, 2000)
voltage = -70 + rng.normal(0, 1.5, size=len(t))
for spike_time in [60, 130]:
    mask = (t >= spike_time) & (t <= spike_time + 2)
    voltage[mask] = 40 * np.exp(-((t[mask] - spike_time - 0.5) ** 2) / 0.2)

n_trials = 20
all_spike_times, all_trial_ids = [], []
for trial in range(n_trials):
    for mean_spike in [60, 130]:
        if rng.random() > 0.15:
            all_spike_times.append(mean_spike + rng.normal(0, 4))
            all_trial_ids.append(trial + 1)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=False)   # NOT shared
axes[0].plot(t, voltage, color="steelblue")
axes[0].set_ylabel("Voltage (mV)")
axes[0].set_title("sharex=False: each panel has its own x-axis (independent zoom)")
axes[0].set_xlabel("Time (ms)")    # now both panels need their own x-axis label
axes[1].scatter(all_spike_times, all_trial_ids, marker="|", s=80, color="black")
axes[1].set_ylabel("Trial")
axes[1].set_xlabel("Time (ms)")
plt.tight_layout()
plt.show()


In [ ]:
# Solution 5-2: add fig.suptitle() to the 4-neuron grid
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=55)
t = np.linspace(0, 150, 1500)
neuron_configs = [
    {"name": "Neuron A (fast)",    "spikes": [30, 60, 90, 120], "noise": 1.5, "color": "steelblue"},
    {"name": "Neuron B (slow)",    "spikes": [75],               "noise": 2.0, "color": "coral"},
    {"name": "Neuron C (bursting)","spikes": [40, 43, 46, 110, 113], "noise": 1.0, "color": "mediumseagreen"},
    {"name": "Neuron D (silent)",  "spikes": [],                 "noise": 3.0, "color": "goldenrod"},
]
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
axes_flat = axes.flatten()
for ax, cfg in zip(axes_flat, neuron_configs):
    v = -70 + rng.normal(0, cfg["noise"], size=len(t))
    for st in cfg["spikes"]:
        mask = (t >= st) & (t <= st + 2)
        v[mask] = 40 * np.exp(-((t[mask] - st - 0.5) ** 2) / 0.2)
    ax.plot(t, v, color=cfg["color"])
    ax.set_title(cfg["name"])
    ax.grid(True, linestyle="--", alpha=0.2)
for ax in axes[1, :]:
    ax.set_xlabel("Time (ms)")
for ax in axes[:, 0]:
    ax.set_ylabel("Voltage (mV)")

fig.suptitle("Four Neurons: Voltage Traces (suptitle appears here)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Solution 5-3: 1x3 grid of sin, cos, sin*cos
import matplotlib.pyplot as plt
import numpy as np
x = np.linspace(0, 4 * np.pi, 400)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True, sharey=True)

axes[0].plot(x, np.sin(x), color="steelblue")
axes[0].set_title("sin(x)")

axes[1].plot(x, np.cos(x), color="coral")
axes[1].set_title("cos(x)")

axes[2].plot(x, np.sin(x) * np.cos(x), color="mediumseagreen")
axes[2].set_title("sin(x) * cos(x)")

for ax in axes:
    ax.set_xlabel("x (radians)")
    ax.grid(True, linestyle="--", alpha=0.3)
axes[0].set_ylabel("Amplitude")

fig.suptitle("Three Trigonometric Functions", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Solution 5-4: equal height_ratios [1, 1] in stimulus/response figure
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=66)
t = np.linspace(0, 300, 3000)
stimulus = np.zeros(len(t))
stimulus[(t >= 50) & (t <= 150)] = 1.0
response_rate = np.zeros(len(t))
response_rate[(t >= 50) & (t <= 150)] = (
    25 * (1 - np.exp(-(t[(t >= 50) & (t <= 150)] - 50) / 20))
)
response_rate[t > 150] = 25 * np.exp(-(t[t > 150] - 150) / 40)
response_rate += rng.normal(0, 1, len(t))
response_rate = np.clip(response_rate, 0, None)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                         gridspec_kw={"height_ratios": [1, 1]})  # equal heights

axes[0].fill_between(t, stimulus, color="orange", alpha=0.7)
axes[0].set_ylabel("Stimulus (on/off)")
axes[0].set_title("Equal panels: stimulus panel is now larger than needed")

axes[1].plot(t, response_rate, color="steelblue")
axes[1].set_ylabel("Firing rate (Hz)")
axes[1].set_xlabel("Time (ms)")

plt.tight_layout()
plt.show()


In [ ]:
# Solution 5-5: add second threshold line at -65 mV to the voltage panel
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=44)
t = np.linspace(0, 200, 2000)
voltage = -70 + rng.normal(0, 1.5, size=len(t))
for spike_time in [60, 130]:
    mask = (t >= spike_time) & (t <= spike_time + 2)
    voltage[mask] = 40 * np.exp(-((t[mask] - spike_time - 0.5) ** 2) / 0.2)
n_trials = 20
all_spike_times, all_trial_ids = [], []
for trial in range(n_trials):
    for mean_spike in [60, 130]:
        if rng.random() > 0.15:
            all_spike_times.append(mean_spike + rng.normal(0, 4))
            all_trial_ids.append(trial + 1)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(t, voltage, color="steelblue", linewidth=1.2)
axes[0].axhline(y=-55, color="crimson", linestyle="--", linewidth=1,
                label="Spike threshold (-55 mV)")
axes[0].axhline(y=-65, color="orange",  linestyle=":",  linewidth=1.5,
                label="Near-threshold depolarisation (-65 mV)")
axes[0].set_ylabel("Voltage (mV)")
axes[0].set_title("Voltage + Raster: two threshold lines")
axes[0].legend(loc="upper right", fontsize=9)

axes[1].scatter(all_spike_times, all_trial_ids, marker="|", s=80, color="black")
axes[1].set_ylabel("Trial")
axes[1].set_xlabel("Time (ms)")

plt.tight_layout()
plt.show()


In [ ]:
# Solution 5-6 (Stretch): 2x2 grid: line, scatter, histogram, text summary panel
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(seed=20)
t = np.linspace(0, 2 * np.pi, 200)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

# Top-left: line plot (sine wave)
axes[0, 0].plot(t, np.sin(t), color="steelblue", linewidth=2)
axes[0, 0].set_title("Line plot: sin(t)")
axes[0, 0].set_xlabel("t")
axes[0, 0].set_ylabel("sin(t)")
axes[0, 0].grid(True, linestyle="--", alpha=0.3)

# Top-right: scatter plot
x_s = rng.normal(0, 1, 80)
y_s = 1.5 * x_s + rng.normal(0, 0.5, 80)
axes[0, 1].scatter(x_s, y_s, color="coral", s=40, alpha=0.7, edgecolors="white")
axes[0, 1].set_title("Scatter: linear relationship")
axes[0, 1].set_xlabel("x")
axes[0, 1].set_ylabel("y")

# Bottom-left: histogram
data = rng.gamma(3, 10, 200)
axes[1, 0].hist(data, bins=20, color="mediumseagreen", edgecolor="white")
axes[1, 0].set_title("Histogram: gamma-distributed data")
axes[1, 0].set_xlabel("Value")
axes[1, 0].set_ylabel("Count")

# Bottom-right: text summary (no data, just a label in the centre)
axes[1, 1].text(0.5, 0.5,
                "Summary panel:
all four plot types in one figure.",
                ha="center", va="center",
                transform=axes[1, 1].transAxes,   # position in axes fractions (0-1)
                fontsize=13, color="dimgrey",
                bbox=dict(facecolor="lightyellow", edgecolor="grey", boxstyle="round,pad=0.5"))
axes[1, 1].set_title("Text panel")
axes[1, 1].set_xticks([])   # hide tick marks (no data here)
axes[1, 1].set_yticks([])

fig.suptitle("Notebook 5 Showcase: Line, Scatter, Histogram, Summary",
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


---
---

## Notebook 5 Complete -- What You Can Now Do

You have covered the five core Matplotlib building blocks. Here is what you have
picked up:

1. **Line plots** -- `plt.plot(x, y)` with figure/axes concepts. You can plot any
   time series, including neural voltage traces and trial-by-trial counts.

2. **Customisation** -- axis labels with units, titles, legends, colours, linestyles,
   linewidths, alpha, axis limits, grids, and threshold lines. Your figures are now
   readable by others.

3. **Scatter plots** -- `plt.scatter()` with per-point colour and size, colour bars,
   and the judgement of when scatter is correct vs line. You can visualise
   relationships between variables and spatial maps of neural data.

4. **Histograms** -- `plt.hist()` with bin choice, density normalisation, and
   overlapping distributions. You can describe and compare how values are
   distributed across a population.

5. **Subplots** -- `plt.subplots(nrows, ncols)`, individual axes addressing, shared
   axes, and `tight_layout()`. You can assemble multi-panel figures like those in
   published neuroscience papers.

**The next notebook (Notebook 6) covers SciPy:** statistics, optimisation, signal
processing, and ordinary differential equations (ODEs) -- the mathematical machinery
that turns plots like these into scientific conclusions.
